# 配套实践 08-02：比较 RNN、LSTM 与 GRU

本练习构造一个延迟记忆任务：机器人早期经历一次“稳定接触”或“滑移警告”，此后当前触觉逐渐回到近零，模型必须在序列末尾判断早先发生了哪种事件。我们将用相同隐藏维度训练 RNN、LSTM 和 GRU，比较学习曲线、不同历史长度上的表现和输入梯度。依赖：PyTorch、NumPy、Matplotlib；Google Colab 的 CPU 即可运行。

<a href="https://qi-robotics.github.io/robot-world-model-tutorial/intermediate/08-time-and-memory/" target="_blank">在新标签页返回课程正文</a>

In [ ]:
import numpy as np  # 整理训练曲线和绘图位置
import matplotlib.pyplot as plt  # 绘制人工序列、准确率和梯度曲线
import torch  # 构造序列张量并训练循环网络
from torch import nn  # 使用 RNN、LSTM、GRU、线性层和损失函数
torch.manual_seed(82)  # 固定模型初始化与训练随机性
np.random.seed(82)  # 固定 NumPy 相关操作的随机性
torch.set_num_threads(1)  # 限制 CPU 线程使免费运行环境更稳定
plt.rcParams["figure.dpi"] = 120  # 提高笔记本图像显示清晰度
plt.rcParams["axes.unicode_minus"] = False  # 避免负号在部分字体中显示异常

## 1. 构造需要记住早期接触的序列

每个时间步有三个输入：触觉事件、末端位置趋势和夹爪开合。触觉事件只在序列早期持续三步，正值表示稳定接触，负值表示滑移警告；后半段两类序列的当前输入非常接近。位置和夹爪信号是与标签无关的干扰项，用来避免任务退化成只读取一个固定数值。

In [ ]:
def make_memory_dataset(sample_count, sequence_length, seed):  # 定义可重复生成延迟记忆数据的函数
    generator = torch.Generator().manual_seed(seed)  # 为当前数据集建立独立随机源
    labels = torch.randint(0, 2, (sample_count,), generator=generator)  # 随机指定滑移警告或稳定接触标签
    event_sign = labels.float() * 2.0 - 1.0  # 把两类标签变成负一和正一触觉事件
    sequences = 0.03 * torch.randn(sample_count, sequence_length, 3, generator=generator)  # 先创建带有小幅传感器噪声的三维序列
    event_start = torch.randint(3, 8, (sample_count,), generator=generator)  # 让关键事件出现在序列前部的不同位置
    for event_offset in range(3):  # 让关键接触事件连续保持三个时间步
        sequences[torch.arange(sample_count), event_start + event_offset, 0] = event_sign  # 写入正负触觉事件
    position_trend = torch.linspace(-0.5, 0.5, sequence_length)  # 构造两类样本共享的末端位置趋势
    sequences[:, :, 1] = position_trend.unsqueeze(0) + 0.02 * torch.randn(sample_count, sequence_length, generator=generator)  # 加入与类别无关的位置变化和噪声
    sequences[:, -8:, 1] = 0.0  # 令序列末尾的位置对两类样本完全相同
    gripper_curve = torch.sigmoid((torch.arange(sequence_length).float() - sequence_length * 0.5) / 2.0)  # 构造逐步闭合且与类别无关的夹爪信号
    sequences[:, :, 2] = gripper_curve.unsqueeze(0)  # 把相同夹爪过程写入全部样本
    return sequences, labels  # 返回形状为样本数乘时间乘特征的输入和类别标签
preview_sequences, preview_labels = make_memory_dataset(80, 40, 820)  # 生成一批长度四十的预览序列
negative_index = int(torch.where(preview_labels == 0)[0][0])  # 找到一个滑移警告样本用于显示
positive_index = int(torch.where(preview_labels == 1)[0][0])  # 找到一个稳定接触样本用于显示
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4), sharey=True)  # 创建两类触觉序列的并排图
axes[0].plot(preview_sequences[negative_index, :, 0], color="tab:orange", marker="o", markersize=3)  # 绘制早期出现负事件的滑移样本
axes[0].set(title="Slip-warning sequence", xlabel="Time step", ylabel="Tactile event value")  # 使用通用英文字体标注滑移警告图的坐标含义
axes[1].plot(preview_sequences[positive_index, :, 0], color="tab:blue", marker="o", markersize=3)  # 绘制早期出现正事件的稳定接触样本
axes[1].set(title="Stable-contact sequence", xlabel="Time step")  # 使用通用英文字体标注稳定接触图的坐标含义
fig.suptitle("The key tactile event appears only near the beginning")  # 使用通用英文字体强调模型必须保留较早信息
fig.tight_layout()  # 调整子图间距避免文字重叠
plt.show()  # 显示两类延迟记忆样本

**怎样理解结果：** 两类序列只在早期触觉事件的方向上不同，末尾触觉都只剩小噪声，位置和夹爪过程也不提供类别答案。模型若只读取最后几步，准确率应接近随机猜测；要正确分类，它必须把早期事件保留到序列末尾。

## 2. 使用统一接口建立三种循环模型

三种模型使用相同的输入维度、隐藏维度和线性分类头。为了让这个小实验中的 LSTM 更容易在训练开始时保留旧信息，我们把遗忘门偏置初始化为 1；这是一种常见初始化选择，不表示所有任务都必须如此设置。

In [ ]:
class SequenceClassifier(nn.Module):  # 定义可切换 RNN、LSTM 和 GRU 的统一分类模型
    def __init__(self, recurrent_kind, input_size=3, hidden_size=24):  # 接收循环单元类型和统一维度配置
        super().__init__()  # 初始化 PyTorch 模块基类
        recurrent_classes = {"RNN": nn.RNN, "LSTM": nn.LSTM, "GRU": nn.GRU}  # 建立名称到循环层类型的映射
        recurrent_class = recurrent_classes[recurrent_kind]  # 根据名称选取本次实验使用的循环层
        self.recurrent = recurrent_class(input_size, hidden_size, batch_first=True)  # 建立按批量时间特征排列的循环层
        self.classifier = nn.Linear(hidden_size, 2)  # 把最后隐藏表示映射为两类事件分数
        if recurrent_kind == "LSTM":  # 只对具有独立遗忘门的 LSTM 执行初始化
            with torch.no_grad():  # 关闭自动求导以安全修改初始偏置
                self.recurrent.bias_ih_l0[hidden_size:2 * hidden_size].fill_(1.0)  # 把第一层输入侧遗忘门偏置设为一
    def forward(self, sequences):  # 定义从整段输入到最终类别分数的前向过程
        hidden_sequence, _ = self.recurrent(sequences)  # 得到每个时间步的隐藏表示并忽略最终状态元组
        final_hidden = hidden_sequence[:, -1, :]  # 读取最后时间步的隐藏表示作为序列摘要
        return self.classifier(final_hidden)  # 输出稳定接触和滑移警告两类分数
model_preview = {kind: SequenceClassifier(kind) for kind in ["RNN", "LSTM", "GRU"]}  # 为三种模型各建立一个结构实例
parameter_counts = {kind: sum(parameter.numel() for parameter in model.parameters()) for kind, model in model_preview.items()}  # 统计三种模型的可训练参数量

## 3. 在同一任务上训练并比较

训练序列长度固定为 40。三个模型使用相同训练集、隐藏维度、优化器和训练轮数，并对梯度进行裁剪。比较结果只说明这一次受控实验中的学习行为，不应被理解成某个模型在所有机器人任务中必然更好。

In [ ]:
def train_sequence_model(recurrent_kind, train_length=40, epochs=60):  # 定义三种循环模型共用的训练流程
    torch.manual_seed(80)  # 让每种模型从可重复的随机状态开始训练
    model = SequenceClassifier(recurrent_kind)  # 建立指定类型的序列分类模型
    optimizer = torch.optim.Adam(model.parameters(), lr=0.005)  # 使用相同学习率的 Adam 优化器
    loss_function = nn.CrossEntropyLoss()  # 使用两类分类的交叉熵损失
    train_inputs, train_labels = make_memory_dataset(1000, train_length, 1)  # 生成固定的训练序列与标签
    test_inputs, test_labels = make_memory_dataset(400, train_length, 2)  # 生成独立随机种子的测试集
    accuracy_history = []  # 创建列表保存每轮训练后的测试准确率
    for epoch in range(epochs):  # 重复多轮小批量参数更新
        shuffled_indices = torch.randperm(len(train_inputs))  # 每轮重新打乱训练样本顺序
        for start in range(0, len(train_inputs), 100):  # 按一百个样本组成训练批次
            batch_indices = shuffled_indices[start:start + 100]  # 取得当前小批量的样本下标
            logits = model(train_inputs[batch_indices])  # 使用当前模型预测整段序列的类别分数
            loss = loss_function(logits, train_labels[batch_indices])  # 计算当前批次分类误差
            optimizer.zero_grad()  # 清除上一批次保留的参数梯度
            loss.backward()  # 通过时间展开计算循环模型的参数梯度
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # 裁剪过大的整体梯度范数
            optimizer.step()  # 根据裁剪后的梯度更新模型参数
        with torch.no_grad():  # 评估阶段关闭梯度以减少额外计算
            predictions = model(test_inputs).argmax(dim=1)  # 取得测试序列中分数最高的类别
            accuracy = (predictions == test_labels).float().mean().item()  # 计算当前轮的测试准确率
        accuracy_history.append(accuracy)  # 保存当前测试结果用于绘制学习曲线
    return model, accuracy_history  # 返回训练模型和完整准确率历史
trained_models = {}  # 创建字典保存三种已经训练的模型
accuracy_histories = {}  # 创建字典保存三种模型的学习曲线
for recurrent_kind in ["RNN", "LSTM", "GRU"]:  # 依次训练三种循环结构
    trained_models[recurrent_kind], accuracy_histories[recurrent_kind] = train_sequence_model(recurrent_kind)  # 使用统一流程完成训练
fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.6))  # 创建准确率曲线与参数量两个坐标轴
for recurrent_kind, history in accuracy_histories.items():  # 依次读取三种模型的测试准确率历史
    axes[0].plot(np.arange(1, len(history) + 1), history, label=recurrent_kind)  # 绘制每轮训练后的测试准确率
axes[0].axhline(0.5, color="black", linestyle="--", linewidth=1, label="Chance")  # 标出二分类随机猜测基线
axes[0].set(title="Learning delayed memory", xlabel="Epoch", ylabel="Test accuracy", ylim=(0.4, 1.03))  # 使用通用英文字体标注学习曲线坐标含义
axes[0].legend()  # 显示三种模型和随机基线的图例
axes[1].bar(parameter_counts.keys(), parameter_counts.values(), color=["#64748b", "#2563eb", "#7c3aed"])  # 绘制相同隐藏维度下的参数量
axes[1].set(title="Parameters at the same hidden size", ylabel="Trainable parameters")  # 使用通用英文字体标注参数量图的含义
fig.tight_layout()  # 自动调整两幅图的间距
plt.show()  # 显示学习曲线与模型规模

**怎样理解结果：** 在固定随机种子的这次实验中，LSTM 较早找到保留触觉事件的方法，RNN 和 GRU 则经过更多训练轮数后才进入高准确率；最终三种模型都可能拟合长度为 40 的任务。右图说明门控结构并非没有代价：LSTM 使用四组门控计算，参数量最多，GRU 位于简单 RNN 与 LSTM 之间。收敛次序会随初始化和数据改变，因此不能据此宣布某种结构永远最好。

## 4. 改变历史长度，检查模型是否只记住了训练节奏

三个模型只见过长度为 40 的训练序列。现在保持事件定义不变，把测试序列改为 20、40、50 和 60 步。如果模型学到的是“保存事件”，它应当在合理范围内适应新的延迟；如果模型依赖固定长度或某种计时捷径，长度变化会暴露问题。

In [ ]:
test_lengths = [20, 40, 50, 60]  # 设置短于、等于和长于训练序列的测试长度
length_accuracies = {kind: [] for kind in trained_models}  # 为每种模型建立不同长度准确率列表
for sequence_length in test_lengths:  # 依次生成四种历史长度的独立测试集
    length_inputs, length_labels = make_memory_dataset(400, sequence_length, 2 + sequence_length)  # 使用与训练不同的随机种子生成测试数据
    for recurrent_kind, model in trained_models.items():  # 依次评估三种已经训练的模型
        with torch.no_grad():  # 关闭梯度以执行纯推理评估
            predictions = model(length_inputs).argmax(dim=1)  # 预测当前历史长度下的接触事件类别
            accuracy = (predictions == length_labels).float().mean().item()  # 计算当前模型和长度的准确率
        length_accuracies[recurrent_kind].append(accuracy)  # 保存结果用于分组柱状图
x_positions = np.arange(len(test_lengths))  # 建立四种测试长度的横轴位置
bar_width = 0.24  # 设置三种模型柱子的宽度
fig, ax = plt.subplots(figsize=(8.5, 3.8))  # 创建不同历史长度的准确率图
for model_index, recurrent_kind in enumerate(["RNN", "LSTM", "GRU"]):  # 依次绘制三种模型的分组柱子
    offset = (model_index - 1) * bar_width  # 计算每种模型相对长度刻度的水平偏移
    ax.bar(x_positions + offset, length_accuracies[recurrent_kind], width=bar_width, label=recurrent_kind)  # 绘制当前模型在四种长度下的准确率
ax.axhline(0.5, color="black", linestyle="--", linewidth=1, label="Chance")  # 标出随机猜测基线
ax.axvline(1.5, color="#94a3b8", linestyle=":", linewidth=1)  # 分隔训练长度附近与更长测试序列
ax.set_xticks(x_positions, [str(length) for length in test_lengths])  # 使用具体时间步数作为横轴标签
ax.set(title="Length generalization after training at T=40", xlabel="Test sequence length", ylabel="Accuracy", ylim=(0.0, 1.05))  # 使用通用英文字体标注长度泛化图的含义
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1.0))  # 把图例放到图外避免遮挡准确率柱子
fig.tight_layout()  # 调整图像边距避免标签被裁切
plt.show()  # 显示不同历史长度下的模型表现

**怎样理解结果：** 长度 40 上的高准确率只说明模型适应了训练分布。简单 RNN 在改变序列长度后可能明显退化，说明它可能同时利用了固定时间节奏，而没有形成稳定的事件记忆；LSTM 和 GRU 在这个随机种子下通常保持得更好。这个结果不是门控网络必然泛化的证明，而是在提醒我们：必须改变依赖距离进行测试，不能只检查训练时使用的固定窗口。

## 5. 查看当前输出对各时间步的局部敏感度

对一个长度为 60 的样本，把正确类别分数对每个输入时间步求梯度，可以观察很小的输入扰动会怎样影响当前输出。梯度不是因果解释，但能辅助判断较早输入是否仍存在有效计算路径。

In [ ]:
probe_inputs, probe_labels = make_memory_dataset(1, 60, 960)  # 生成一个比训练序列更长的探测样本
gradient_profiles = {}  # 创建字典保存三种模型对各时间步的输入梯度
for recurrent_kind, model in trained_models.items():  # 依次分析每一种已经训练的循环模型
    differentiable_input = probe_inputs.clone().requires_grad_(True)  # 复制输入并允许计算输入梯度
    model.zero_grad()  # 清除模型中可能残留的梯度
    correct_score = model(differentiable_input)[0, probe_labels.item()]  # 取得真实类别对应的模型分数
    correct_score.backward()  # 计算该分数对全部输入时间步的梯度
    gradient_norm = differentiable_input.grad[0].norm(dim=1).detach().numpy()  # 汇总每个时间步三个输入维度的梯度范数
    gradient_profiles[recurrent_kind] = gradient_norm  # 保存当前模型的时间梯度曲线
fig, ax = plt.subplots(figsize=(8.5, 3.8))  # 创建时间梯度对比图
for recurrent_kind, gradient_norm in gradient_profiles.items():  # 依次绘制三种模型的梯度范数
    ax.semilogy(np.arange(len(gradient_norm)), gradient_norm + 1e-10, label=recurrent_kind)  # 使用对数纵轴显示跨数量级的梯度
ax.axvspan(3, 10, color="#fef3c7", alpha=0.7, label="Possible event region")  # 标出数据生成时关键触觉事件的早期范围
ax.set(title="Local sensitivity of the final output over time", xlabel="Input time step", ylabel="Input-gradient norm (log scale)")  # 使用通用英文字体标注梯度图的含义
ax.legend()  # 显示三种模型和事件区域的图例
fig.tight_layout()  # 调整图像边距避免轴标签被裁切
plt.show()  # 显示输入梯度随时间位置的变化

**怎样理解结果：** 如果早期区域的梯度已经接近数值零，当前输出很难通过局部更新响应早期输入；若仍有明显梯度，说明至少存在一条可传播影响的路径。梯度大小会受到饱和、模型置信度和单个样本影响，不能单独用来证明模型理解了接触事件，必须与长度泛化、顺序打乱和模态删除等实验结合。

**本练习的结论：** 门控结构为长期信息提供了更可控的更新路径，但它们并不会自动获得正确记忆。数据时间对齐、任务是否真正需要历史、训练长度以及评测方式都会改变结果。下一章的 Attention 将保留多个历史表示，并让当前查询直接选择需要读取的位置。